### **This code compute the 2, 5, 10, 20, and 40 years return period for Peru based on PISCOp daily data (MAPs) (1981-2023)**
 
**Created:** 10/29/2024 by Jorge Mayo (jmayo@rti.org)  
**Project #:** 0219481  
**Last modified:** 02/20/2025 by Jorge Mayo  
**Status:** completed and debugged, cleaning up to do  
**QA Status:** reviewed by  
**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2-Analisis_Vulnerabilidad\AI2c_AnalisisBlueSpo\Rainfall Analysis and  
Research Triangle Institute\IKI Peru Project - General\Interno\AI2c_AnalisisBlueSpot\Scripts
 
**Objective:** create csv and a shapefile with the 2, 5, 10, 20 and 40 return period for Peru using the MAP stored in a sqlite database (Wateralloc structure)  
**Compatibility:** The code could be slightly modify to use csv or another MAP files instead of sqlite databases  
**Packages:** numpy, pandas, geopandas, sqlite3, scipy  
**Further documentation:**  
 
**Inputs:** basins shapefile, MAP from gridded precipitacion stored in a sqlite database.   
**Outputs:** csv file, shapefile file wit the return periods for the specified years.
 
**Assumptions:** assumptions on script operations (e.g., inputs in subfolder of wd) or on calculations or theories used in code
 
**Future work:** 
 
**Notes:** The function calculate_and_fill_return_periods function was created to compute return periods using Weibull method and fill missing data based on the proximity of those basins (use this one).

In [1]:
import numpy as np
import pandas as pd
import sqlite3
import geopandas as gpd
from scipy.spatial import cKDTree

ModuleNotFoundError: No module named 'scipy'

In [ ]:
#user = 'jmayo'
user= 'sgilson'
#user = 

In [7]:
subbasins_shapefile = f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - Interno/AI2b_Modelacion/Reparametrizacion_AHD/Shapefiles/New_Extent/Peru_AHD_clipped.shp'
#f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Reparametrizacion_AHD/Shapefiles/New_Extent/Peru_AHD_clipped.shp'
subbasins_gdf = gpd.read_file(subbasins_shapefile).set_index('COMID').to_crs('WGS84')
subbasins_gdf.drop(columns=['Cuenca', 'Obser'], inplace=True)

In [8]:
subbasins_gdf

,GRID_CODE,GRID_COUNT,PROD_UNIT,AREASQKM,geometry
COMID,,,,,
311153600,311152200,588,3,120.898,"POLYGON ((-69.91667 -16.0375, -69.91667 -16.01..."
311092600,311096100,289,3,59.448,"POLYGON ((-69.85417 -15.8875, -69.85417 -15.89..."
310914100,310877900,717,3,147.843,"POLYGON ((-70.05833 -15.50833, -70.05833 -15.5..."
310832400,310821000,435,3,89.735,"POLYGON ((-70.0125 -15.27917, -70.00833 -15.27..."
310892700,310892100,424,3,87.429,"POLYGON ((-70.09583 -15.4625, -70.10833 -15.46..."
...,...,...,...,...,...
320259100,319237900,653,3,132.279,"POLYGON ((-81.0125 -4.05, -81.0125 -4.04583, -..."
320259200,319238000,43,3,8.397,"POLYGON ((-80.85833 -3.89326, -80.85833 -3.908..."
320259300,319238100,689,3,140.504,"POLYGON ((-80.73122 -3.71002, -80.73175 -3.713..."


### Using calculate_and_fill_return_periods fills the nan values with the most proximity COMID with data.

In [13]:
db_path = f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - Interno/AI2b_Modelacion/Clima/BD_PISCO/Peru_PISCO_MAP_Clipped.sqlite'
#f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Clima/BD_PISCO/Peru_PISCO_MAP_Clipped.sqlite'
table_name = 'catchment_met_observations'
return_periods = [2, 5, 10, 20, 40]

In [18]:
# Function to calculate return periods and fill missing data based on proximity
def calculate_and_fill_return_periods(db_path, table_name, gdf, return_periods, target_crs="EPSG:32718"):
    # Convert GeoDataFrame to the specified projected CRS for accurate distance calculations
    gdf_projected = gdf.to_crs(target_crs)
    
    # Initialize an empty list to store results
    results = []
    
    # Loop through each COMID from the index of gdf
    for comid in gdf_projected.index:
        # Extract data for the current COMID
        df_comid_daily = extract_comid_time_series(db_path, table_name, comid)
        
        # Check if the dataframe is empty
        if df_comid_daily.empty:
            print(f"No data available for COMID {comid}. Skipping.")
            continue
        
        # Convert 'measured_date' to datetime
        df_comid_daily['measured_date'] = pd.to_datetime(df_comid_daily['measured_date'])
        
        # Drop any rows with NaN in 'avg_precip_cm'
        df_comid_daily = df_comid_daily.dropna(subset=['avg_precip_cm'])

        # Obtener la precipitación máxima anual
        df_comid = df_comid_daily.groupby(df_comid_daily['measured_date'].dt.year)['avg_precip_cm'].max().reset_index()
        
        # Sort by 'avg_precip_cm' in descending order
        df_sorted = df_comid.sort_values(by='avg_precip_cm', ascending=False).reset_index(drop=True)
        
        # Rank and calculate '% exceedence' and 'return period'
        df_sorted['rank'] = df_sorted['avg_precip_cm'].rank(method='first', ascending=False)
        df_sorted['% exceedence'] = df_sorted['rank'] / (len(df_sorted) + 1)
        
        # Calculate return periods years, handling NaN or inf
        df_sorted['return period (years)'] = 1 / df_sorted['% exceedence']

        # Calculate the values for specified return periods
        return_period_values = {'COMID': comid}
        for period in return_periods:
            # Find the row with return period closest to the desired period
            closest_row = df_sorted.iloc[(df_sorted['return period (years)'] - period).abs().argsort()[:1]]
            avg_precip_value = closest_row['avg_precip_cm'].values[0] if not closest_row.empty else np.nan
            return_period_values[f'{period}_yrs_cm'] = avg_precip_value
        
        # Append the results for the current COMID
        results.append(return_period_values)
    
    # Convert results to a DataFrame
    df_results = pd.DataFrame(results).set_index('COMID')
    
    # Merge with the original GeoDataFrame (in the projected CRS)
    gdf_results = gdf_projected.join(df_results, how='left')
    
    # Find basins with missing return period values
    empty_basins = gdf_results[gdf_results.isna().any(axis=1)]
    valid_basins = gdf_results.dropna()
    
    # Use spatial proximity to fill missing return period values
    if not empty_basins.empty:
        # Get coordinates for empty and valid basins
        empty_coords = np.array(list(empty_basins.geometry.centroid.apply(lambda geom: (geom.x, geom.y))))
        valid_coords = np.array(list(valid_basins.geometry.centroid.apply(lambda geom: (geom.x, geom.y))))
        
        # Build a KDTree for nearest neighbor search
        tree = cKDTree(valid_coords)
        distances, indices = tree.query(empty_coords)
        
        # Assign values from nearest valid basin
        for i, idx in enumerate(empty_basins.index):
            nearest_idx = valid_basins.index[indices[i]]
            gdf_results.loc[idx, df_results.columns] = gdf_results.loc[nearest_idx, df_results.columns]
    
    # Convert back to original CRS if needed
    gdf_results = gdf_results.to_crs(gdf.crs)
    
    # Save to CSV and shapefile
    gdf_results[df_results.columns].to_csv('return_periods_filled_Peru.csv')
    gdf_results.to_file('return_periods_filled_Peru.shp', driver='ESRI Shapefile')
    
    return gdf_results

# Supporting function to extract data for each COMID
def extract_comid_time_series(db_path, table_name, comid):
    conn = sqlite3.connect(db_path)
    query = f"""
    SELECT * 
    FROM {table_name}
    WHERE COMID = ?
    ORDER BY measured_date
    """
    df = pd.read_sql_query(query, conn, params=(comid,))
    conn.close()
    return df

In [19]:
# Assuming you have a GeoDataFrame `gdf` with COMID as the index
gdf_with_filled_return_periods = calculate_and_fill_return_periods(db_path, table_name, subbasins_gdf, return_periods)

In [20]:
gdf_with_filled_return_periods

,GRID_CODE,GRID_COUNT,PROD_UNIT,AREASQKM,geometry,2_yrs_cm,5_yrs_cm,10_yrs_cm,20_yrs_cm,40_yrs_cm
COMID,,,,,,,,,,
311153600,311152200,588,3,120.898,"POLYGON ((-69.91667 -16.0375, -69.91667 -16.01...",2.378,3.184,3.331,3.986,4.656
311092600,311096100,289,3,59.448,"POLYGON ((-69.85417 -15.8875, -69.85417 -15.89...",2.294,2.815,3.227,3.436,3.918
310914100,310877900,717,3,147.843,"POLYGON ((-70.05833 -15.50833, -70.05833 -15.5...",1.350,1.712,1.869,2.265,2.591
310832400,310821000,435,3,89.735,"POLYGON ((-70.0125 -15.27917, -70.00833 -15.27...",1.519,1.802,1.959,2.163,4.591
310892700,310892100,424,3,87.429,"POLYGON ((-70.09583 -15.4625, -70.10833 -15.46...",1.542,1.972,2.143,2.189,2.368
...,...,...,...,...,...,...,...,...,...,...
320259100,319237900,653,3,132.279,"POLYGON ((-81.0125 -4.05, -81.0125 -4.04583, -...",0.707,1.319,2.041,2.446,3.368
320259200,319238000,43,3,8.397,"POLYGON ((-80.85833 -3.89326, -80.85833 -3.908...",0.916,1.797,2.208,4.087,5.594
320259300,319238100,689,3,140.504,"POLYGON ((-80.73122 -3.71002, -80.73175 -3.713...",1.386,2.737,3.294,4.567,6.818
